# 08 Validera

Innan vi analyserar kontrollerar vi att den sammanslagna tabellen håller. Det kallas att validera. Vi skriver ner vad vi förväntar oss av datan som regler och låter koden pröva varje regel. En regel som inte håller betyder att något har gått fel någonstans på vägen, i en källa eller i vår egen kod.

Reglerna är av fyra slag.

- **Struktur.** Rätt antal rader och unika nycklar.
- **Format och intervall.** Koder har rätt antal siffror och värden ligger inom rimliga gränser.
- **Fullständighet.** Värden saknas bara där vi vet att de ska saknas.
- **Samstämmighet.** Uppgifter som borde hänga ihop gör det, även mellan olika källor.

In [ ]:
import json
from pathlib import Path

import pandas as pd

RADATA = Path("..") / "data" / "radata"
HARMONISERAD = Path("..") / "data" / "harmoniserad"

webbplatser = pd.read_parquet(HARMONISERAD / "webbplatser.parquet")

webbplatser.shape

## Ett sätt att samla kontroller

Vi skriver en funktion som tar ett namn och ett villkor. Villkoret är `True` om regeln håller. Funktionen sparar resultatet i en lista, så att vi kan se alla kontroller tillsammans på slutet.

In [ ]:
kontroller = []


def kontrollera(namn, villkor):
    godkand = bool(villkor)
    kontroller.append({"kontroll": namn, "godkand": godkand})
    print("OK  " if godkand else "FEL ", namn)

`bool` gör om villkoret till `True` eller `False`. Funktionen skriver ut OK eller FEL direkt, så att vi ser resultatet medan vi går igenom notebooken.

## Struktur

In [ ]:
ar_kommun = webbplatser["typ"] == "kommun"
ar_region = webbplatser["typ"] == "region"

kontrollera("289 kommuner", ar_kommun.sum() == 289)
kontrollera("21 regioner", ar_region.sum() == 21)

for nyckel in ["webperf_id", "qid", "origin", "kolada_id"]:
    kontrollera(f"{nyckel} är unik", webbplatser[nyckel].is_unique)

## Format

Kommunkoder och Koladas id har fyra siffror och länskoder två. Med `str.fullmatch` prövar vi om hela texten följer ett mönster. `\d` betyder en siffra och `{4}` att den ska upprepas fyra gånger. Ett sådant mönster kallas reguljärt uttryck.

Kommunkoden saknas för regionerna, så den kontrollerar vi bara där den finns. `dropna()` tar bort tomma värden.

In [ ]:
kontrollera("kommunkod har fyra siffror", webbplatser["kommunkod"].dropna().str.fullmatch(r"\d{4}").all())
kontrollera("kolada_id har fyra siffror", webbplatser["kolada_id"].str.fullmatch(r"\d{4}").all())
kontrollera("lanskod har två siffror", webbplatser["lanskod"].str.fullmatch(r"\d{2}").all())
kontrollera("origin börjar med https://", webbplatser["origin"].str.startswith("https://").all())

`r` före en text betyder att bakstreck ska läsas som de står. Det behövs ofta i reguljära uttryck.

## Intervall

Webperfs betyg går från 1 till 5. Tider i millisekunder och bredbandshastighet ska vara större än noll, och CLS kan inte vara negativt. `between` prövar om ett värde ligger mellan två gränser, gränserna inräknade.

In [ ]:
betyg = ["betyg_totalt", "betyg_tillganglighet", "betyg_prestanda", "betyg_sakerhet", "betyg_standard"]

for kolumn in betyg:
    kontrollera(f"{kolumn} mellan 1 och 5", webbplatser[kolumn].dropna().between(1, 5).all())

kontrollera("lcp_p75_ms större än noll", (webbplatser["lcp_p75_ms"].dropna() > 0).all())
kontrollera("inp_p75_ms större än noll", (webbplatser["inp_p75_ms"].dropna() > 0).all())
kontrollera("cls_p75 inte negativ", (webbplatser["cls_p75"].dropna() >= 0).all())
kontrollera("bredband större än noll", (webbplatser["bredband_ned_mbit_s"] > 0).all())
kontrollera("befolkning större än noll", (webbplatser["befolkning"] > 0).all())

### Fångar kontrollen ett fel?

En kontroll som aldrig slår larm kan lika gärna vara trasig. Vi provar därför på en kopia av tabellen där vi med flit sätter ett betyg till 7. `copy` gör en kopia, så att vi inte ändrar den riktiga tabellen. Resultatet skriver vi bara ut och sparar inte bland kontrollerna.

In [ ]:
prov = webbplatser.copy()
prov.loc[0, "betyg_totalt"] = 7

prov["betyg_totalt"].between(1, 5).all()

`False` betyder att kontrollen hade upptäckt felet.

## Fullständighet

Vissa värden vet vi saknas. Regionerna har ingen kommunkod, utom Gotland som också är kommun. Webbplatser som saknas i CrUX har inga mått. Vi kontrollerar att värden saknas just där och ingen annanstans.

In [ ]:
saknar_kommunkod = webbplatser["kommunkod"].isna()

kontrollera("bara regioner saknar kommunkod", (ar_region | ~saknar_kommunkod).all())
kontrollera("alla kommuner har kommunkod", (~saknar_kommunkod[ar_kommun]).all())

har_crux = webbplatser["crux_status"] == 200
kontrollera("cls finns precis för dem som har CrUX-data", (webbplatser["cls_p75"].notna() == har_crux).all())

for kolumn in ["befolkning", "invanare_kolada", "bredband_ned_mbit_s", "lan"] + betyg:
    kontrollera(f"{kolumn} saknas inte", webbplatser[kolumn].notna().all())

`|` betyder eller. Regeln `ar_region | ~saknar_kommunkod` är sann för en rad som antingen är en region eller har en kommunkod.

Vi kontrollerar CLS och inte LCP eller INP, eftersom de två kan saknas även när CrUX har data, när för få besökare har mätts.

## Samstämmighet

### Kommunkod och län

De två första siffrorna i en kommunkod är länskoden. Kommunkoden kommer från ett påstående i Wikidata och länet från ett annat, så det här kontrollerar att de två stämmer med varandra.

In [ ]:
med_kod = webbplatser[webbplatser["kommunkod"].notna()]

kontrollera("kommunkoden börjar med länskoden", (med_kod["kommunkod"].str[:2] == med_kod["lanskod"]).all())

### Befolkning från två källor

Vi har befolkningen från både Wikidata och Kolada. De mäter vid olika tidpunkter, Wikidata 30 juni 2026 och Kolada 31 december 2025, så de behöver inte vara exakt lika. Vi räknar ut skillnaden i procent av Koladas värde.

In [ ]:
skillnad = (webbplatser["befolkning"] - webbplatser["invanare_kolada"]) / webbplatser["invanare_kolada"] * 100

skillnad.describe()

`describe` sammanfattar en kolumn med antal, medelvärde, spridning och de minsta och största värdena.

På ett halvår förändras befolkningen sällan mer än någon procent. Vi sätter gränsen vid 2 procent. En större skillnad skulle tyda på att en källa avser något annat område eller en annan tid än vi tror. Vi visar också de största skillnaderna åt båda håll.

In [ ]:
kontrollera("befolkning skiljer högst 2 procent mellan Wikidata och Kolada", (skillnad.abs() <= 2).all())

webbplatser.assign(skillnad_procent=skillnad.round(2)).sort_values("skillnad_procent")[
    ["namn", "befolkning", "invanare_kolada", "skillnad_procent"]
].iloc[[0, 1, 2, -3, -2, -1]]

`assign` lägger till en kolumn i en kopia av tabellen och `sort_values` sorterar efter den.

### Summan blir hela Sverige

Regionerna täcker tillsammans hela landet, så summan av regionernas befolkning ska vara nära Sveriges. Sveriges befolkning finns i Kolada under id `0000`, som vi tog bort när vi harmoniserade. Vi hämtar den ur rådatan.

In [ ]:
kolada_radata = json.loads((RADATA / "kolada-N01951-2025.json").read_text(encoding="utf-8"))

for post in kolada_radata["svar"]["values"]:
    if post["municipality"] == "0000":
        for varde in post["values"]:
            if varde["gender"] == "T":
                sverige = varde["value"]

summa_regioner = webbplatser.loc[ar_region, "befolkning"].sum()

sverige, summa_regioner, round((summa_regioner - sverige) / sverige * 100, 2)

In [ ]:
kontrollera("regionernas befolkning skiljer högst 1 procent från Sveriges", abs(summa_regioner - sverige) / sverige <= 0.01)

### Samma mätperiod

Alla svar från CrUX ska gälla samma 28 dagar. `nunique` räknar antalet olika värden och bortser från tomma.

In [ ]:
kontrollera("CrUX har en enda mätperiod", webbplatser["period_slut"].nunique() == 1)

## Sammanfattning

Vi samlar alla kontroller i en tabell och räknar hur många som gick igenom.

In [ ]:
resultat = pd.DataFrame(kontroller)

resultat["godkand"].value_counts()

Om någon kontroll inte går igenom ska vi inte gå vidare till analysen. Därför avslutar vi med en rad som stoppar med ett fel och listar de kontroller som misslyckades.

In [ ]:
misslyckade = resultat.loc[~resultat["godkand"], "kontroll"].tolist()

if misslyckade:
    raise ValueError(f"Kontroller som inte gick igenom: {misslyckade}")

print("Alla", len(resultat), "kontroller gick igenom")

## Det här vet vi nu

- Validering betyder att vi skriver ner vad vi förväntar oss av datan och låter koden pröva det.
- En kontroll bör provas mot ett känt fel, så att vi vet att den fungerar.
- Saknade värden ska finnas där vi väntar oss dem och ingen annanstans.
- Källor som mäter samma sak kan kontrollera varandra, men skillnader i tidpunkt måste vi räkna med.
- Om valideringen inte går igenom stannar vi innan analysen.

I nästa notebook lagrar vi datan så att den är lätt att hitta, fråga och dela.